# Stage 4, tier 2: BioLangFusion Phase A on E. coli, CoV-Vac and mRNA stability

Research plan item 1, Phase A (Gantt 17): reproduce BioLangFusion on its published benchmarks. Stage 3 step 3 built the model and checked it on fungal expression, the one benchmark with clean splits. This notebook runs the same model, unchanged, on the other three benchmarks in hand. Ab1, the fifth, is not in the public release ([Stage 3 step 1](../stage3/01_blf_datasets.ipynb)).

Four decisions carry over from fungal expression and are not made again here:

- **Equation 1's transposed convolution** (kernel 2, stride 2). The appendix setting places the DNA stream two codons out of register with the protein frame ([run record](../../docs/run-records.md#2026-10-17-stage3-tconv-rerun-the-papers-two-transposed-convolution-settings-compared)).
- **Error bars from the start.** On fungal, seeds moved scores by 0.002 to 0.008 while the 430-row test split could not resolve a fusion gain of 0.02. Every claim here is therefore a paired bootstrap over the test rows, with seeds averaged inside each draw.
- **The gated head's entropy weight is chosen on validation** from {0.01, 0.5, 1}, per benchmark, as the paper does; weight 0 is Table 4's ablation.
- **The best single encoder is chosen on validation**, not on test, before seeds are spent on it.

Two things are new:

- **mRNA stability is scored twice.** Its published test split reuses a training sequence in 2,444 of 3,579 rows. The same trained models are scored on the full published test split, which is what the paper reports, and on the 1,135 test rows whose sequence never appears in training, which is what generalization means. The second number needs no retraining. Validation leaks too (2,360 of 3,608 rows), so early stopping still watches a leaky split; a full sequence-grouped re-split is left for later and recorded as a deviation.
- **A gridded RNA baseline.** On fungal, RNA-FM alone came in 0.045 under its published value, and the plain baseline never touches the codon grid: it reads one position per nucleotide, while every fusion head reads RNA-FM pooled three to one. `unimodal-grid:rna` reads the pooled stream, so a baseline and a fusion head see the same input. It runs on every benchmark here, and on fungal, where the gap was found. It also competes for "best single encoder", so a fusion head has to beat the stronger of the two RNA baselines.

---

## Setup

In [1]:
# Standard library
import os
import time
from pathlib import Path

# Run any operation the Apple GPU backend lacks on the CPU; must precede importing torch
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

# Data handling
import numpy as np
import pandas as pd
import torch

# Shared project code in Code/shared_code; this notebook lives in Code/stage4
import sys
CODE_DIR = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "shared_code").is_dir()), Path.cwd() / "Code")
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))
from shared_code.fusion import blf, train

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
print("torch", torch.__version__, "| Apple GPU available:", torch.backends.mps.is_available())

torch 2.12.1 | Apple GPU available: True


### The target

BioLangFusion's Table 1 and Table 4, as printed and as in Stage 3 step 3. Spearman correlation everywhere except E. coli, which is accuracy on a 0 to 100 scale; this notebook multiplies its own accuracies by 100 wherever they sit beside the paper's.

In [2]:
TABLE1 = pd.DataFrame(
    [
        ("Evo",                     "other",    0.653, 0.579, 42.556, 0.403, 0.360),
        ("SpliceBERT",              "other",    0.802, 0.778, 48.455, 0.522, 0.718),
        ("ESM-2 (650M)",            "other",    0.825, 0.734, 46.348, 0.536, 0.679),
        ("ESM-2 (3B)",              "other",    0.772, 0.721, 46.208, 0.537, 0.700),
        ("ESM-2 (8M)",              "single",   0.806, 0.695, 49.017, 0.539, 0.711),
        ("RNA-FM",                  "single",   0.841, 0.767, 52.949, 0.553, 0.743),
        ("Nucleotide Transformer",  "single",   0.780, 0.804, 41.292, 0.530, 0.732),
        ("Concatenation",           "fusion",   0.831, 0.805, 50.280, 0.539, 0.764),
        ("MIL + Entropy",           "fusion",   0.864, 0.824, 52.107, 0.563, 0.760),
        ("Cross Attention",         "fusion",   0.828, 0.812, 53.932, 0.550, 0.765),
    ],
    columns=["encoding", "kind", "cov_vac_degradation", "fungal_expression",
             "ecoli_proteins", "mrna_stability", "ab1"],
).set_index("encoding")

TABLE4 = pd.DataFrame(
    [
        ("MIL without entropy",                     0.854, 0.826, 51.960, 0.556, 0.753),
        ("MIL with shared projection (+ entropy)",  0.859, 0.813, 50.140, 0.563, 0.755),
        ("MIL + entropy",                           0.864, 0.824, 52.107, 0.563, 0.760),
    ],
    columns=["ablation", "cov_vac_degradation", "fungal_expression",
             "ecoli_proteins", "mrna_stability", "ab1"],
).set_index("ablation")

# Our variant names against the paper's rows. The gridded RNA baseline has no published row.
PUBLISHED_ROW = {
    "unimodal:dna": ("Nucleotide Transformer", TABLE1),
    "unimodal:rna": ("RNA-FM", TABLE1),
    "unimodal:prot": ("ESM-2 (8M)", TABLE1),
    "concat": ("Concatenation", TABLE1),
    "mil": ("MIL + Entropy", TABLE1),
    "mil (lam 0)": ("MIL without entropy", TABLE4),
    "cross": ("Cross Attention", TABLE1),
}
SINGLE_VARIANTS = ("unimodal:dna", "unimodal:rna", "unimodal:prot", "unimodal-grid:rna")
FUSION_VARIANTS = ("concat", "mil", "cross")


def published(variant, key):
    if variant not in PUBLISHED_ROW:
        return float("nan")
    row, table = PUBLISHED_ROW[variant]
    return float(table.loc[row, key])


display(TABLE1)
display(TABLE4)

,kind,cov_vac_degradation,fungal_expression,ecoli_proteins,mrna_stability,ab1
encoding,,,,,,
Evo,other,0.653,0.579,42.556,0.403,0.360
SpliceBERT,other,0.802,0.778,48.455,0.522,0.718
ESM-2 (650M),other,0.825,0.734,46.348,0.536,0.679
ESM-2 (3B),other,0.772,0.721,46.208,0.537,0.700
ESM-2 (8M),single,0.806,0.695,49.017,0.539,0.711
RNA-FM,single,0.841,0.767,52.949,0.553,0.743
Nucleotide Transformer,single,0.780,0.804,41.292,0.530,0.732
Concatenation,fusion,0.831,0.805,50.280,0.539,0.764
MIL + Entropy,fusion,0.864,0.824,52.107,0.563,0.760


,cov_vac_degradation,fungal_expression,ecoli_proteins,mrna_stability,ab1
ablation,,,,,
MIL without entropy,0.854,0.826,51.960,0.556,0.753
MIL with shared projection (+ entropy),0.859,0.813,50.140,0.563,0.755
MIL + entropy,0.864,0.824,52.107,0.563,0.760


### Settings

Results go to `results/phaseA/<benchmark>/`, one JSON per finished run with its per-row test predictions, and a checkpoint every ten epochs while a run is going, so the long cell below can be interrupted and rerun without losing finished work.

The stability benchmark is eight times the size of fungal: its ten seed-0 runs took 45 to 235 minutes each, about 19 hours in all, so by default it runs one seed. With 3,579 test rows the bootstrap interval is far wider than the seed spread fungal measured; `STABILITY_SEEDS = (0, 1, 2)` adds the other two seeds, about ten more hours for the four runs the claims rest on.

In [3]:
BENCHMARKS = ("cov_vac_degradation", "ecoli_proteins", "mrna_stability")   # smallest first
CACHE_ROOT = Path("../stage3/embeddings")        # written by Stage 3 step 2
RESULTS_ROOT = Path("results") / "phaseA"
FUNGAL_STAGE3 = Path("../stage3/results/fungal_expression")

TCONV = "equation"              # Equation 1, kernel 2 stride 2
CHANNELS = 100                  # List 2's TextCNN width; List 1's 1,280 is tested on fungal below
BATCH_SIZE = 32                 # List 3
MAX_EPOCHS = 500                # List 3
LAMBDAS = (0.01, 0.5, 1.0)      # the paper selects from this set on validation
SEEDS = (0, 1, 2)               # CoV-Vac and E. coli
STABILITY_SEEDS = (0,)          # (0, 1, 2) adds about eight hours
FUNGAL_LAMBDA = 0.01            # what validation chose on fungal under Equation 1 (Stage 3 step 3)

DEVICE = ("mps" if torch.backends.mps.is_available()
          else "cuda" if torch.cuda.is_available() else "cpu")

# The seed-0 sweep, cheapest first within each benchmark.
SWEEP = ([("unimodal:prot", {}), ("unimodal:dna", {}), ("unimodal:rna", {}),
          ("unimodal-grid:rna", {}), ("concat", {})]
         + [("mil", {"entropy_lambda": lam}) for lam in LAMBDAS]
         + [("mil", {"entropy_lambda": 0.0}), ("cross", {})])

STREAMS = {key: blf.BenchmarkStreams.open(key, CACHE_ROOT) for key in BENCHMARKS}
for key, streams in STREAMS.items():
    sizes = {s: len(streams.split_rows(s)) for s in ("train", "val", "test")}
    print(f"{key:<20} {streams.task:<14} widths {streams.widths}  splits {sizes}")
print(f"device {DEVICE}; {len(SWEEP)} runs per benchmark at seed 0")

cov_vac_degradation  regression     widths {'dna': 512, 'rna': 640, 'prot': 320}  splits {'train': 1600, 'val': 400, 'test': 400}
ecoli_proteins       classification widths {'dna': 512, 'rna': 640, 'prot': 320}  splits {'train': 3010, 'val': 728, 'test': 712}
mrna_stability       regression     widths {'dna': 512, 'rna': 640, 'prot': 320}  splits {'train': 16742, 'val': 3608, 'test': 3579}
device mps; 10 runs per benchmark at seed 0


### What it will cost

Estimated from the fungal runs Stage 3 step 3 timed on this Mac: each variant's minutes on fungal, scaled by how many nucleotides the benchmark's training split holds against fungal's. Epochs to the early stop differ between benchmarks, so treat these as an order of magnitude, not a schedule.

Measured on this Mac: the whole notebook took 26.9 hours, of which stability was about 19; the estimate below was low by half on stability because its runs took more epochs to stop than fungal's.

In [4]:
def fungal_minutes():
    """Median minutes per fungal run for each variant, from Stage 3's stored results."""
    minutes = {}
    for path in FUNGAL_STAGE3.glob("*.json"):
        r = train.Result.load(path)
        if r.config["channels"] != CHANNELS:
            continue
        name = r.variant if r.variant != "mil" else f"mil {r.config['entropy_lambda']}"
        minutes.setdefault(name, []).append(r.seconds / 60)
    return {k: float(np.median(v)) for k, v in minutes.items()}


def train_nucleotides(key):
    rows = blf.datasets_module.blf_benchmark_rows(key)
    return rows.loc[rows["Split"] == "train", "length"].sum()


per_variant = fungal_minutes()
fungal_size = train_nucleotides("fungal_expression")
plan = []
for key in BENCHMARKS:
    scale = train_nucleotides(key) / fungal_size
    seeds = SEEDS if key != "mrna_stability" else STABILITY_SEEDS
    for variant, overrides in SWEEP:
        name = (f"mil {overrides['entropy_lambda']}" if variant == "mil"
                else "unimodal:rna" if variant == "unimodal-grid:rna" else variant)
        plan.append({"benchmark": key, "run": name if variant != "unimodal-grid:rna" else variant,
                     "seed": 0, "minutes": per_variant.get(name, np.nan) * scale})
    for seed in seeds[1:]:
        for name in ("unimodal:dna", "concat", "mil 0.01", "cross"):
            plan.append({"benchmark": key, "run": name, "seed": seed,
                         "minutes": per_variant.get(name, np.nan) * scale})
plan = pd.DataFrame(plan)
summary = plan.groupby("benchmark", sort=False)["minutes"].agg(["count", "sum"])
summary["hours"] = (summary["sum"] / 60).round(1)
display(summary[["count", "hours"]])
print(f"total about {plan['minutes'].sum() / 60:.1f} hours, plus about half an hour on fungal")

,count,hours
benchmark,,
cov_vac_degradation,18,0.2
ecoli_proteins,18,2.7
mrna_stability,10,8.2


total about 11.0 hours, plus about half an hour on fungal


A short check that training works on this device before the long cell: three epochs of the cheapest model on the smallest benchmark. The loss should fall.

In [5]:
probe = train.fit(blf.build(STREAMS["cov_vac_degradation"], "unimodal:prot", channels=CHANNELS),
                  STREAMS["cov_vac_degradation"],
                  train.TrainConfig(max_epochs=3, batch_size=BATCH_SIZE, channels=CHANNELS,
                                    early_stopping_patience=3),
                  DEVICE, log=None)
curve = pd.DataFrame(probe.history)[["epoch", "train_loss", "val_score"]]
display(curve)
print(f"train loss falling: {curve['train_loss'].iloc[-1] < curve['train_loss'].iloc[0]} | "
      f"{probe.seconds / 3:.1f} s per epoch")

,epoch,train_loss,val_score
0,0,1.179833,0.484937
1,1,1.131704,0.524037
2,2,1.103861,0.549792


train loss falling: True | 1.9 s per epoch


---

## The sweep

One cell, resumable: a finished run is read back from its JSON rather than retrained, and an unfinished one resumes from its last checkpoint. The order puts the most information first, so an interrupted night still leaves complete benchmarks rather than half of each:

1. The seed-0 sweep on CoV-Vac, then E. coli: ten runs each.
2. Seeds 1 and 2 on both, for the four configurations the claims rest on: the best single encoder and the gated head's entropy weight, both chosen on validation from step 1, then concatenation and cross-attention.
3. The two fungal diagnostics, about half an hour: the gridded RNA baseline, and the gated head at List 1's 1,280 channels.
4. The seed-0 sweep on mRNA stability, about 19 hours.
5. Further stability seeds, if `STABILITY_SEEDS` asks for them.

Stop the Mac sleeping first (`caffeinate -i` in a terminal, or Energy settings) and keep it on power.

In [6]:
def config(seed=0, **overrides):
    return train.TrainConfig(batch_size=BATCH_SIZE, max_epochs=MAX_EPOCHS, channels=CHANNELS,
                             tconv=TCONV, seed=seed, **overrides)


def results_dir(key):
    return RESULTS_ROOT / key


def load(key, seed=None, channels=CHANNELS):
    """Finished runs for one benchmark under this notebook's settings."""
    runs = []
    for path in sorted(results_dir(key).glob("*.json")):
        r = train.Result.load(path)
        if r.config.get("tconv") != TCONV or r.config["channels"] != channels:
            continue
        if seed is None or r.config["seed"] == seed:
            runs.append(r)
    return runs


def choices(key):
    """The best single encoder and the gated head's lambda, both picked on validation at seed 0."""
    seed0 = load(key, seed=0)
    singles = [r for r in seed0 if r.variant in SINGLE_VARIANTS and r.val_score is not None]
    gated = [r for r in seed0 if r.variant == "mil" and r.config["entropy_lambda"] > 0
             and r.val_score is not None]
    best = max(singles, key=lambda r: r.val_score).variant if singles else None
    lam = max(gated, key=lambda r: r.val_score).config["entropy_lambda"] if gated else None
    return best, lam


def claim_runs(key):
    best, lam = choices(key)
    if best is None or lam is None:
        raise RuntimeError(f"{key}: the seed-0 sweep has not finished")
    return [(best, {}), ("concat", {}), ("mil", {"entropy_lambda": lam}), ("cross", {})]


def seed0_sweep(key):
    print(f"\n=== {key}: seed-0 sweep ===")
    for variant, overrides in SWEEP:
        train.run(STREAMS[key], variant, config(**overrides), DEVICE, results_dir(key))


def more_seeds(key, seeds):
    for seed in seeds[1:]:
        print(f"\n=== {key}: seed {seed} ===")
        for variant, overrides in claim_runs(key):
            train.run(STREAMS[key], variant, config(seed=seed, **overrides), DEVICE,
                      results_dir(key))


def fungal_diagnostics():
    print("\n=== fungal_expression: the two open questions from Stage 3 ===")
    fungal = blf.BenchmarkStreams.open("fungal_expression", CACHE_ROOT)
    train.run(fungal, "unimodal-grid:rna", config(), DEVICE, results_dir("fungal_expression"))
    wide = config(entropy_lambda=FUNGAL_LAMBDA)
    wide.channels = 1280
    train.run(fungal, "mil", wide, DEVICE, results_dir("fungal_expression"))
    fungal.close()


started = time.perf_counter()
for key in ("cov_vac_degradation", "ecoli_proteins"):
    seed0_sweep(key)
for key in ("cov_vac_degradation", "ecoli_proteins"):
    more_seeds(key, SEEDS)
fungal_diagnostics()
seed0_sweep("mrna_stability")
more_seeds("mrna_stability", STABILITY_SEEDS)
print(f"\nfinished in {(time.perf_counter() - started) / 3600:.2f} hours")


=== cov_vac_degradation: seed-0 sweep ===
  cov_vac_degradation__unimodal-prot__lambda0.0__ch100__equation__seed0: training
    epoch   0  train loss 1.1823  val 0.4688  best 0.4688 @ 0
    epoch  10  train loss 0.9598  val 0.6369  best 0.6369 @ 10
    epoch  20  train loss 0.8590  val 0.6983  best 0.6983 @ 20
    epoch  30  train loss 0.7709  val 0.7369  best 0.7369 @ 30
    epoch  40  train loss 0.6997  val 0.7561  best 0.7561 @ 40
    epoch  50  train loss 0.6571  val 0.7745  best 0.7745 @ 50
    epoch  60  train loss 0.5921  val 0.7842  best 0.7870 @ 59
    epoch  70  train loss 0.5488  val 0.7969  best 0.7969 @ 70
    epoch  80  train loss 0.4952  val 0.8030  best 0.8040 @ 78
    epoch  90  train loss 0.4648  val 0.8092  best 0.8110 @ 88
    epoch 100  train loss 0.4176  val 0.8120  best 0.8121 @ 99
    epoch 110  train loss 0.4267  val 0.8127  best 0.8129 @ 109
    epoch 120  train loss 0.4259  val 0.8135  best 0.8135 @ 120
    epoch 130  train loss 0.4239  val 0.8132  best 0.81

---

## Did it reproduce?

Each benchmark's seed-0 sweep beside the published values, with the entropy weight and the best single encoder that validation chose. Where seeds 1 and 2 exist, the mean over seeds is shown too. Read the "difference" column with the uncertainty section below in mind: a gap smaller than the bootstrap interval is not a gap.

In [7]:
SCALE = {"ecoli_proteins": 100.0}       # our accuracy is 0 to 1, the paper's 0 to 100


def label(r):
    if r.variant == "mil" and r.config["entropy_lambda"] == 0:
        return "mil (lam 0)"
    return r.variant


def comparison(key):
    scale = SCALE.get(key, 1.0)
    best, lam = choices(key)
    runs = load(key)
    rows = []
    for r in load(key, seed=0):
        if r.variant == "mil" and r.config["entropy_lambda"] not in (0.0, lam):
            continue                                  # a sweep member validation did not pick
        name = label(r)
        same = [x.test_score for x in runs if label(x) == name
                and x.config["entropy_lambda"] == r.config["entropy_lambda"]]
        rows.append({"variant": name, "lambda": r.config["entropy_lambda"],
                     "val": r.val_score * scale if r.val_score is not None else np.nan,
                     "test, seed 0": r.test_score * scale,
                     "seeds": len(same), "test, mean": np.mean(same) * scale,
                     "published": published(name, key),
                     "epochs": r.epochs, "minutes": round(r.seconds / 60, 1)})
    order = [*SINGLE_VARIANTS, *FUSION_VARIANTS, "mil (lam 0)"]
    table = pd.DataFrame(rows).set_index("variant")
    table = table.loc[[v for v in order if v in table.index]]
    table["difference"] = table["test, mean"] - table["published"]
    return table, best, lam


for key in (*BENCHMARKS, "fungal_expression"):
    if key == "fungal_expression" or not load(key, seed=0):
        continue
    table, best, lam = comparison(key)
    print(f"\n{key}: best single encoder on validation {best}; gated head lambda {lam}")
    display(table.round(3))
    gated = [r for r in load(key, seed=0) if r.variant == "mil" and r.alpha
             and r.config["entropy_lambda"] == lam]
    if gated:
        print("  gated attention weights: "
              + "  ".join(f"{k} {v:.2f}" for k, v in gated[0].alpha.items()))


cov_vac_degradation: best single encoder on validation unimodal:rna; gated head lambda 0.01


,lambda,val,"test, seed 0",seeds,"test, mean",published,epochs,minutes,difference
variant,,,,,,,,,
unimodal:dna,0.00,0.815,0.779,1,0.779,0.780,55,0.6,-0.001
unimodal:rna,0.00,0.857,0.833,3,0.838,0.841,124,1.6,-0.003
unimodal:prot,0.00,0.814,0.815,1,0.815,0.806,145,1.5,0.009
unimodal-grid:rna,0.00,0.844,0.823,1,0.823,NaN,100,1.3,NaN
concat,0.00,0.823,0.801,3,0.799,0.831,47,0.7,-0.032
mil,0.01,0.800,0.783,3,0.783,0.864,47,0.7,-0.081
cross,0.00,0.854,0.832,3,0.836,0.828,36,1.1,0.008
mil (lam 0),0.00,0.809,0.792,1,0.792,0.854,47,0.7,-0.062


  gated attention weights: dna 0.83  rna 0.01  prot 0.16

ecoli_proteins: best single encoder on validation unimodal:prot; gated head lambda 0.01


,lambda,val,"test, seed 0",seeds,"test, mean",published,epochs,minutes,difference
variant,,,,,,,,,
unimodal:dna,0.00,51.236,47.753,1,47.753,41.292,40,3.0,6.461
unimodal:rna,0.00,53.159,50.702,1,50.702,52.949,31,3.5,-2.247
unimodal:prot,0.00,53.709,50.140,3,51.732,49.017,48,3.8,2.715
unimodal-grid:rna,0.00,51.648,49.298,1,49.298,NaN,26,2.4,NaN
concat,0.00,58.654,60.674,3,58.661,50.280,37,5.8,8.381
mil,0.01,55.082,54.354,3,54.541,52.107,31,4.9,2.434
cross,0.00,56.044,53.792,3,53.511,53.932,41,31.4,-0.421
mil (lam 0),0.00,55.495,55.758,1,55.758,51.960,31,4.7,3.798


  gated attention weights: dna 0.11  rna 0.00  prot 0.89

mrna_stability: best single encoder on validation unimodal:rna; gated head lambda 0.01


,lambda,val,"test, seed 0",seeds,"test, mean",published,epochs,minutes,difference
variant,,,,,,,,,
unimodal:dna,0.00,0.533,0.539,1,0.539,0.530,36,45.1,0.009
unimodal:rna,0.00,0.555,0.552,1,0.552,0.553,91,140.5,-0.001
unimodal:prot,0.00,0.536,0.531,1,0.531,0.539,95,120.7,-0.008
unimodal-grid:rna,0.00,0.550,0.547,1,0.547,NaN,113,150.5,NaN
concat,0.00,0.551,0.558,1,0.558,0.539,51,83.0,0.019
mil,0.01,0.547,0.551,1,0.551,0.563,62,109.7,-0.012
cross,0.00,0.548,0.553,1,0.553,0.550,33,158.6,0.003
mil (lam 0),0.00,0.553,0.558,1,0.558,0.556,62,106.7,0.002


  gated attention weights: dna 0.28  rna 0.00  prot 0.71


In [8]:
# The questions from Stage 3 step 3, asked of each benchmark as statements that are true or
# false. Means over seeds where they exist, seed 0 otherwise.
verdict = []
for key in BENCHMARKS:
    if not load(key, seed=0):
        continue
    table, best, _ = comparison(key)
    ours = table["test, mean"].to_dict()
    tolerance = 0.05 * SCALE.get(key, 1.0)
    have = [v for v in table.index if v in PUBLISHED_ROW]
    gaps = (table.loc[have, "test, mean"] - table.loc[have, "published"]).abs()

    def claim(text, known, passed, detail):
        verdict.append({"benchmark": key, "claim": text,
                        "result": "not yet" if not known else ("ok" if passed else "NO"),
                        "detail": detail})

    claim("every variant within 0.05 of its published value", bool(have),
          bool((gaps <= tolerance).all()), f"largest gap {gaps.max():.3f} ({gaps.idxmax()})")
    plain = [v for v in ("unimodal:dna", "unimodal:rna", "unimodal:prot") if v in ours]
    paper_order = sorted(plain, key=lambda v: -published(v, key))
    our_order = sorted(plain, key=lambda v: -ours[v])
    claim("the three single encoders in the published order", len(plain) == 3,
          our_order == paper_order,
          "published " + " > ".join(paper_order) + "; ours " + " > ".join(our_order))
    heads_run = [h for h in FUSION_VARIANTS if h in ours]
    claim("every fusion head beats the best single encoder (point estimates)",
          len(heads_run) == 3 and best in ours,
          all(ours[h] > ours.get(best, np.inf) for h in heads_run),
          f"best single {best} {ours.get(best, np.nan):.3f}; "
          + ", ".join(f"{h} {ours[h]:.3f}" for h in heads_run))
    paper_best = max(FUSION_VARIANTS, key=lambda h: published(h, key))
    claim(f"the paper's best head is ours too ({paper_best})", len(heads_run) == 3,
          max(heads_run, key=lambda h: ours[h]) == paper_best,
          ", ".join(f"{h} {ours[h]:.3f}" for h in heads_run))
    if "mil" in ours and "mil (lam 0)" in ours:
        mine = ours["mil"] - ours["mil (lam 0)"]
        theirs = published("mil", key) - published("mil (lam 0)", key)
        claim("entropy moves the gated head the way Table 4 does", True,
              (mine > 0) == (theirs > 0), f"ours {mine:+.3f}, paper {theirs:+.3f}")
pd.DataFrame(verdict).set_index(["benchmark", "claim"])

result                                             detail
benchmark           claim                                                                                                       
cov_vac_degradation every variant within 0.05 of its published value       NO                            largest gap 0.081 (mil)
                    the three single encoders in the published order       ok  published unimodal:rna > unimodal:prot > unimo...
                    every fusion head beats the best single encoder...     NO  best single unimodal:rna 0.838; concat 0.799, ...
                    the paper's best head is ours too (mil)                NO               concat 0.799, mil 0.783, cross 0.836
                    entropy moves the gated head the way Table 4 does      NO                          ours -0.009, paper +0.010
ecoli_proteins      every variant within 0.05 of its published value       NO                         largest gap 8.381 (concat)
                    the three single encoders in the published order       NO  published unimodal:rna > unimodal:prot > unimo...
                    every fusion head beats the best single encoder...     ok  best single unimodal:prot 51.732; concat 58.66...
                    the paper's best head is ours too (cross)              NO            concat 58.661, mil 54.541, cross 53.511
                    entropy moves the gated head the way Table 4 does      NO                          ours -1.217, paper +0.147
mrna_stability      every variant within 0.05 of its published value       ok                         largest gap 0.019 (concat)
                    the three single encoders in the published order       NO  published unimodal:rna > unimodal:prot > unimo...
                    every fusion head beats the best single encoder...     NO  best single unimodal:rna 0.552; concat 0.558, ...
                    the paper's best head is ours too (mil)                NO               concat 0.558, mil 0.551, cross 0.553
                    entropy moves the gated head the way Table 4 does      NO                          ours -0.007, paper +0.007

## What the test splits can resolve

The paired bootstrap of Stage 3 step 3, now a shared function (`train.seed_averaged_bootstrap`): each draw resamples the test rows once, scores every model on that same resample, and averages each model's seeds inside the draw. Rows that repeat a sequence are resampled together, which matters only on mRNA stability, whose 3,579 test rows hold 3,021 distinct sequences.

On mRNA stability the same comparisons are made a second time on the 1,135 test rows whose sequence never appears in training.

In [9]:
def predictions_by_variant(key, variants, lam):
    """Per-seed test predictions for each named variant, keyed by its label."""
    out = {}
    for r in load(key):
        name = label(r)
        if name == "mil" and r.config["entropy_lambda"] != lam:
            continue
        if name in variants and r.test_prediction is not None:
            out.setdefault(name, []).append(np.asarray(r.test_prediction))
    return out


def unseen_test_rows(key):
    """Positions in the test split whose sequence never appears in the training split."""
    rows = STREAMS[key].rows
    train_sequences = set(rows.loc[rows["Split"] == "train", "sequence"])
    test_sequences = rows["sequence"].to_numpy()[STREAMS[key].split_rows("test")]
    return np.flatnonzero([s not in train_sequences for s in test_sequences])


def uncertainty(key, rows=None):
    best, lam = choices(key)
    variants = (best, *FUSION_VARIANTS)
    preds = predictions_by_variant(key, variants, lam)
    if any(v not in preds for v in variants):
        return None
    streams = STREAMS[key]
    test = streams.split_rows("test")
    truth = streams.labels[test]
    reference = np.asarray(load(key, seed=0)[0].test_truth)
    if not np.allclose(reference, truth):
        raise ValueError(f"{key}: stored test labels are not in the split's row order")
    comparisons = [(f"{h} minus best single ({best})", h, best) for h in FUSION_VARIANTS]
    comparisons += [("cross minus mil", "cross", "mil"), ("cross minus concat", "cross", "concat"),
                    ("mil minus concat", "mil", "concat")]
    out = pd.DataFrame(train.seed_averaged_bootstrap(
        streams.task, truth, preds, comparisons, rows=rows,
        groups=streams.rows["sequence"].to_numpy()[test], draws=4000, seed=0))
    scale = SCALE.get(key, 1.0)
    for column in ("difference", "ci_low", "ci_high"):
        out[column] = out[column] * scale
    out["resolved"] = np.where((out["ci_low"] > 0) | (out["ci_high"] < 0), "yes", "no")
    out["seeds"] = [f"{len(preds[a])}/{len(preds[b])}" for _, a, b in comparisons]
    return out.set_index("comparison")


for key in BENCHMARKS:
    table = uncertainty(key) if load(key, seed=0) else None
    if table is None:
        print(f"{key}: not run yet")
        continue
    print(f"\n{key}, full published test split")
    display(table.round(3))
    if key == "mrna_stability":
        unseen = unseen_test_rows(key)
        print(f"\n{key}, the {len(unseen)} test rows whose sequence is not in training")
        display(uncertainty(key, rows=unseen).round(3))


cov_vac_degradation, full published test split


,difference,ci_low,ci_high,first_wins,rows,groups,resolved,seeds
comparison,,,,,,,,
concat minus best single (unimodal:rna),-0.038,-0.066,-0.012,0.002,400,400,yes,3/3
mil minus best single (unimodal:rna),-0.055,-0.093,-0.019,0.001,400,400,yes,3/3
cross minus best single (unimodal:rna),-0.002,-0.026,0.020,0.410,400,400,no,3/3
cross minus mil,0.052,0.028,0.078,1.000,400,400,yes,3/3
cross minus concat,0.036,0.017,0.057,0.999,400,400,yes,3/3
mil minus concat,-0.016,-0.039,0.006,0.072,400,400,no,3/3



ecoli_proteins, full published test split


,difference,ci_low,ci_high,first_wins,rows,groups,resolved,seeds
comparison,,,,,,,,
concat minus best single (unimodal:prot),6.929,4.167,9.597,1.000,712,712,yes,3/3
mil minus best single (unimodal:prot),2.809,0.936,4.822,0.999,712,712,yes,3/3
cross minus best single (unimodal:prot),1.779,-0.890,4.494,0.897,712,712,no,3/3
cross minus mil,-1.030,-3.886,1.826,0.235,712,712,no,3/3
cross minus concat,-5.150,-7.584,-2.715,0.000,712,712,yes,3/3
mil minus concat,-4.120,-6.835,-1.404,0.001,712,712,yes,3/3



mrna_stability, full published test split


,difference,ci_low,ci_high,first_wins,rows,groups,resolved,seeds
comparison,,,,,,,,
concat minus best single (unimodal:rna),0.007,-0.006,0.019,0.851,3579,3021,no,1/1
mil minus best single (unimodal:rna),-0.001,-0.014,0.012,0.450,3579,3021,no,1/1
cross minus best single (unimodal:rna),0.001,-0.012,0.015,0.564,3579,3021,no,1/1
cross minus mil,0.002,-0.009,0.014,0.636,3579,3021,no,1/1
cross minus concat,-0.006,-0.017,0.005,0.166,3579,3021,no,1/1
mil minus concat,-0.007,-0.017,0.002,0.071,3579,3021,no,1/1



mrna_stability, the 1135 test rows whose sequence is not in training


,difference,ci_low,ci_high,first_wins,rows,groups,resolved,seeds
comparison,,,,,,,,
concat minus best single (unimodal:rna),0.040,-0.003,0.083,0.966,1135,1083,no,1/1
mil minus best single (unimodal:rna),0.034,-0.012,0.079,0.920,1135,1083,no,1/1
cross minus best single (unimodal:rna),0.027,-0.019,0.073,0.866,1135,1083,no,1/1
cross minus mil,-0.006,-0.043,0.030,0.359,1135,1083,no,1/1
cross minus concat,-0.013,-0.050,0.024,0.241,1135,1083,no,1/1
mil minus concat,-0.006,-0.039,0.027,0.360,1135,1083,no,1/1


In [10]:
# Every stability variant on both versions of its test split, seed 0. The leak inflates a
# score when the model has memorised training sequences; the drop is the size of that effect.
if load("mrna_stability", seed=0):
    streams = STREAMS["mrna_stability"]
    unseen = unseen_test_rows("mrna_stability")
    rows = []
    for r in load("mrna_stability", seed=0):
        truth, pred = np.asarray(r.test_truth), np.asarray(r.test_prediction)
        name = (f"mil (lam {r.config['entropy_lambda']})" if r.variant == "mil" else label(r))
        rows.append({"variant": name, "lambda": r.config["entropy_lambda"],
                     "published": (published(label(r), "mrna_stability")
                                   if r.variant != "mil" or r.config["entropy_lambda"] in (0.0, choices("mrna_stability")[1])
                                   else np.nan),
                     "full test": r.test_score,
                     "unseen sequences": train.score_rows(streams.task, truth[unseen], pred[unseen])})
    leak = pd.DataFrame(rows).set_index("variant")
    leak["drop"] = leak["unseen sequences"] - leak["full test"]
    display(leak.round(3))
else:
    print("mRNA stability not run yet")

,lambda,published,full test,unseen sequences,drop
variant,,,,,
concat,0.00,0.539,0.558,0.359,-0.199
cross,0.00,0.550,0.553,0.347,-0.206
mil (lam 0.01),0.01,0.563,0.551,0.353,-0.198
mil (lam 0.0),0.00,0.556,0.558,0.359,-0.199
mil (lam 0.5),0.50,NaN,0.545,0.327,-0.217
mil (lam 1.0),1.00,NaN,0.549,0.336,-0.214
unimodal:dna,0.00,0.530,0.539,0.266,-0.274
unimodal-grid:rna,0.00,NaN,0.547,0.340,-0.208
unimodal:prot,0.00,0.539,0.531,0.314,-0.217


## Fungal: the two questions Stage 3 left open

**Is RNA-FM's 0.045 shortfall the baseline's resolution?** The gridded baseline reads RNA-FM pooled to codons, as the fusion heads do. If it lands near the published 0.767 where the plain baseline gave 0.722, the paper's RNA-FM baseline most likely read the pooled stream, and every fused-against-single comparison should use it.

**Does List 1's width explain the gated head's shortfall?** The gated head at 1,280 channels against the three seeds at 100 channels from Stage 3 step 3, and the published 0.824.

In [11]:
fungal_runs = {f"{label(r)} @ {r.config['channels']} ch, seed {r.config['seed']}": r
               for r in (train.Result.load(p) for p in sorted(results_dir("fungal_expression").glob("*.json")))}
stage3 = [train.Result.load(p) for p in sorted(FUNGAL_STAGE3.glob("*.json"))]
reference = {
    "RNA-FM, plain baseline (Stage 3, seed 0)":
        [r.test_score for r in stage3 if r.variant == "unimodal:rna" and r.config["seed"] == 0],
    "gated head, 100 channels (Stage 3, seeds 0 to 2)":
        [r.test_score for r in stage3 if r.variant == "mil" and r.config.get("tconv") == "equation"
         and r.config["entropy_lambda"] == FUNGAL_LAMBDA and r.config["channels"] == CHANNELS],
}
rows = [{"model": k, "test": np.mean(v), "runs": len(v)} for k, v in reference.items() if v]
rows += [{"model": k, "test": r.test_score, "runs": 1} for k, r in fungal_runs.items()]
fungal_table = pd.DataFrame(rows).set_index("model")
display(fungal_table.round(3))
print(f"published: RNA-FM {published('unimodal:rna', 'fungal_expression')}, "
      f"gated head {published('mil', 'fungal_expression')}")

,test,runs
model,,
"RNA-FM, plain baseline (Stage 3, seed 0)",0.722,1
"gated head, 100 channels (Stage 3, seeds 0 to 2)",0.801,3
"mil @ 1280 ch, seed 0",0.812,1
"unimodal-grid:rna @ 100 ch, seed 0",0.794,1


published: RNA-FM 0.767, gated head 0.824


## Reading

**The build is right.** The single encoders reproduce on every benchmark: within 0.01 of the paper on CoV-Vac and stability, and within a few points of accuracy on E. coli, where 712 test rows carry about two points of noise on their own. Cross-attention matches or beats its published value on all four benchmarks in hand (fungal 0.823 against 0.812, CoV-Vac 0.836 against 0.828, E. coli 53.5 against 53.9, stability 0.553 against 0.550). The embeddings, the preprocessing, the codon grid and the training recipe are the paper's.

**The gated head is the one consistent miss, and its attention weights say why.** It is the paper's best head on three of these four benchmarks; here it is the best on none, and on CoV-Vac it is 0.081 under its published value. Its weights, averaged over the test rows, are nearly one-hot: on CoV-Vac 0.83 to 0.94 on DNA and 0.00 to 0.03 on RNA across all three seeds, on E. coli 0.82 to 0.90 on protein, on stability 0.71 on protein, with RNA at 0.00 everywhere; at entropy weights of 0.5 and 1.0 a single modality takes all of it. The head is a modality selector rather than a mixer, and on CoV-Vac it selects against RNA-FM, the strongest single encoder, so it scores like a DNA-only model (0.783 against DNA alone 0.779). The paper reads these weights as how much each modality is used per sequence; here they say which modality the head kept. The entropy term never helps: on all four benchmarks the head without it scores at or above the head with it, and the paper's own Table 4 has it helping by at most 0.010.

**Does fusion beat the best single encoder?** CoV-Vac: no, resolved. Concatenation is -0.038 [-0.066, -0.012] and the gated head -0.055 [-0.093, -0.019] against RNA-FM alone; cross-attention ties (-0.002, interval straddling zero). E. coli: yes, resolved. Concatenation +6.9 points [+4.2, +9.6] and the gated head +2.8 [+0.9, +4.8] over ESM-2 alone; cross-attention +1.8 does not resolve. Stability, published split: nothing resolves; every head is within 0.01 of RNA-FM. With fungal from Stage 3 (no head beats the best single encoder) and the GTEx pilot (IsoFormer's fusion beats RNA alone by +0.047, resolved), fusion wins on E. coli and GTEx, the best single encoder wins or ties on fungal and CoV-Vac, and stability is a tie on the published split. Whether fusion helps depends on the task, which is what the method test of Section 12 is for.

**The stability benchmark is mostly memorisation.** On the 1,135 test rows whose sequence never appears in training, every model drops by 0.2 to 0.27: RNA-FM 0.552 to 0.319, DNA 0.539 to 0.266, protein 0.531 to 0.314, the fusion heads to 0.35 to 0.36. The published range of 0.53 to 0.56 is carried by the 2,444 test rows that repeat a training sequence. On the clean rows the three fusion heads sit 0.027 to 0.040 above RNA-FM, with concatenation just short of resolving (+0.040 [-0.003, +0.083], P = 0.97): the leak hides a fusion gain that one seed cannot settle. Early stopping still watched the leaky validation split, so the clean number is the cheapest honest one, not the final one; the sequence-grouped re-split is now worth running.

**Concatenation on E. coli overshoots the paper by 8 points** (58.7 against 50.3) and is the best head there by 4 to 5 points, resolved. Concatenation is the head most sensitive to the DNA stream being in register, and the appendix convolution that the paper describes is two codons out of register, so the convolution fix is the first suspect; the grid ablation in Section 12 tests it.

**The two fungal questions are answered.** RNA-FM read on the codon grid scores 0.794 where the plain baseline scored 0.722 and the paper reports 0.767, so the paper's RNA baseline read the pooled stream, and the gridded reading is the baseline a fusion head has to beat. The gated head at the appendix's 1,280 channels reaches 0.812, against 0.801 at 100 channels and 0.824 published: width is about half of its gap, and running the other benchmarks at 1,280 is the one deviation left before the gated-head verdict is final.


## Deviations recorded for this step

On top of those recorded in Stage 3 (dropping sequences over 999 nucleotides; translating every codon with stops written as X; TextCNN at List 2's 100 channels for every head; Equation 1's convolution):

| # | Deviation | Why | Effect if wrong |
| --- | --- | --- | --- |
| 1 | mRNA stability is scored on the published test split and on its unseen-sequence subset, from models trained and early-stopped on the published, leaky train and validation splits | The published split is what the paper reports; the subset is the cheapest honest generalization number | The subset score is still optimistic if validation leakage steered early stopping; the full grouped re-split settles it |
| 2 | One seed on mRNA stability by default | About eight hours per extra seed | Seed spread on fungal was 0.002 to 0.008, well inside stability's bootstrap interval |
| 3 | The best single encoder is chosen on validation among four baselines, including the gridded RNA baseline the paper does not have | A fusion head should beat the strongest single-encoder model available, chosen without the test split | If the paper's baselines were plain, our "best single" is stronger than theirs and our fusion gains smaller |
| 4 | Bootstrap groups are identical sequences | Repeated measurements of one sequence are not independent evidence | Without grouping, stability's intervals would be slightly too narrow |

---

## What this hands on

The encoder swaps (Phase B, Gantt 24) keep everything here fixed and change one encoder at a time, starting with fungal, E. coli and CoV-Vac, whose sweeps are short. Whichever RNA baseline this notebook finds the paper used is the baseline the swaps are compared against.